In [1]:
# ============================================================
# Notebook : 02_eda.ipynb
# Project  : Causal-Guided Multimodal Sensor Fusion for Elderly Risk Detection
# Purpose  : Exploratory data analysis supporting the modelling choices:
#            label distribution, lagged cross-correlation between sensors and
#            the risk target, risk-event hour-of-day distribution, the
#            CO2-TVOC correlation-vs-direction illustration, and the inter-
#            sensor correlation structure.
#            All figures are grayscale, 600 dpi, saved as .png and .pdf,
#            without in-figure captions (captions live in the manuscript).
# Note     : Anonymous repository for double-blind review.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ── Paths & grayscale publication style ──────────────────────
ROOT     = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"
FIG_DIR  = ROOT / "results" / "figures"
TAB_DIR  = ROOT / "results" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True); TAB_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")
mpl.rcParams.update({"figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
                     "font.family": "DejaVu Sans", "axes.unicode_minus": False,
                     "image.cmap": "Greys", "axes.grid": True, "grid.alpha": 0.3})
def savefig(fig, name):
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
GC = {"d": "#222222", "m": "#777777", "l": "#BBBBBB"}   # gray shades

FEAT = ['Activity_IR', 'HeartRate', 'BreathRate', 'StressIndex',
        'Temperature', 'Humidity', 'CO2', 'TVOC']
LABEL_MAP = {'위험': 1, '주의': 1, '수면': 0, '식사': 0, '외출': 0, '기타': 0}
EN = {'위험': 'Danger', '주의': 'Caution', '수면': 'Sleep',
      '식사': 'Meal', '외출': 'Outing', '기타': 'Other'}

df = pd.read_pickle(PROC_DIR / "df_new.pkl")
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df['label'] = df['Estimation'].map(LABEL_MAP)
df = df[df['label'].notna()].copy()
df.sort_values(['HH_ID', 'Timestamp'], inplace=True)
df['Hour'] = df['Timestamp'].dt.hour

# ── Fig 1. Label distribution (6-class and binary) ───────────
order = ['기타', '주의', '수면', '외출', '수면']  # placeholder, recomputed below
cnt = df['Estimation'].value_counts()
cats = ['위험', '주의', '수면', '식사', '외출', '기타']
cnt = cnt.reindex(cats).fillna(0)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
grays = [str(v) for v in np.linspace(0.15, 0.80, len(cats))]
ax[0].bar([EN[c] for c in cats], cnt.values, color=grays, edgecolor='black', linewidth=0.6)
ax[0].set_ylabel('Record count'); ax[0].tick_params(axis='x', rotation=30)
binc = df['label'].value_counts().reindex([0, 1]).fillna(0)
ax[1].bar(['Normal (0)', 'Risk (1)'], binc.values, color=[GC['l'], GC['d']],
          edgecolor='black', linewidth=0.6)
for i, v in enumerate(binc.values):
    ax[1].text(i, v, f"{v/binc.sum()*100:.1f}%", ha='center', va='bottom', fontsize=9)
ax[1].set_ylabel('Record count')
savefig(fig, "fig01_label_distribution")

# ── Fig 2. Lagged cross-correlation (motivation for temporal causality) ──
df['Risk_bin'] = df['label'].astype(float)
def cross_corr(x, y, max_lag=6):
    out = {}
    x = x.values; y = y.values
    for lag in range(max_lag + 1):
        xi = x[:len(x) - lag] if lag > 0 else x
        yi = y[lag:] if lag > 0 else y
        m = ~(np.isnan(xi) | np.isnan(yi))
        out[lag] = stats.pearsonr(xi[m], yi[m]) if m.sum() > 10 else (np.nan, np.nan)
    return out
PAIRS = [('Activity_IR', 'HeartRate'), ('Activity_IR', 'Risk_bin'),
         ('HeartRate', 'Risk_bin'), ('CO2', 'TVOC')]
LAB = ['Activity_IR -> HeartRate', 'Activity_IR -> Risk',
       'HeartRate -> Risk', 'CO2 -> TVOC']
fig, axes = plt.subplots(2, 2, figsize=(9, 6))
ccf_rows = []
for ax_, (cx, cy), lab in zip(axes.flatten(), PAIRS, LAB):
    res = cross_corr(df[cx], df[cy], 6)
    lags = list(res.keys()); rs = [res[l][0] for l in lags]; ps = [res[l][1] for l in lags]
    bars = ax_.bar(lags, rs, color=GC['m'], edgecolor='black', linewidth=0.6, width=0.6)
    for b, p in zip(bars, ps):
        if p is not None and not np.isnan(p) and p < 0.05:
            b.set_color(GC['d'])
    ax_.axhline(0, color='black', lw=0.8, ls='--')
    ax_.set_xlabel('Lag (x 10 min)'); ax_.set_ylabel('Pearson r'); ax_.set_title(lab, fontsize=9)
    for l in lags:
        ccf_rows.append({'pair': lab, 'lag': l, 'r': round(res[l][0], 4), 'p': res[l][1]})
savefig(fig, "fig02_lag_cross_correlation")
pd.DataFrame(ccf_rows).to_csv(TAB_DIR / "eda_lag_cross_correlation.csv", index=False)

# ── Fig 3. Risk-event hour-of-day distribution ───────────────
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax_, est, shade in [(axes[0], '위험', GC['d']), (axes[1], '주의', GC['m'])]:
    h = df.loc[df['Estimation'] == est, 'Hour'].value_counts().reindex(range(24), fill_value=0)
    ax_.bar(range(24), h.values, color=shade, edgecolor='black', linewidth=0.4, width=0.85)
    ax_.axvspan(-0.5, 5.5, color='0.5', alpha=0.15)
    ax_.set_xlim(-0.5, 23.5); ax_.set_xticks(range(0, 24, 3))
    ax_.set_xlabel('Hour of day'); ax_.set_ylabel('Count'); ax_.set_title(EN[est], fontsize=9)
savefig(fig, "fig03_risk_by_hour")

# ── Fig 4. CO2 vs TVOC : correlation vs direction ────────────
pair = df[['CO2', 'TVOC']].dropna()
r, p = stats.pearsonr(pair['CO2'], pair['TVOC'])
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].scatter(pair['CO2'], pair['TVOC'], s=3, alpha=0.05, color=GC['d'], rasterized=True)
axes[0].set_xlabel('CO2'); axes[0].set_ylabel('TVOC'); axes[0].set_title(f"r = {r:.2f}", fontsize=9)
fwd = cross_corr(df['CO2'], df['TVOC'], 6); bwd = cross_corr(df['TVOC'], df['CO2'], 6)
lags = list(fwd.keys())
axes[1].plot(lags, [fwd[l][0] for l in lags], 'o-', color=GC['d'], label='CO2 leads TVOC')
axes[1].plot(lags, [bwd[l][0] for l in lags], 's--', color=GC['m'], label='TVOC leads CO2')
axes[1].set_xlabel('Lag (x 10 min)'); axes[1].set_ylabel('Pearson r'); axes[1].legend(fontsize=8)
savefig(fig, "fig04_co2_tvoc")

# ── Fig 5. Inter-sensor correlation heatmap (grayscale) ──────
corr = df[FEAT].corr()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Greys", vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white', square=True, cbar_kws={'label': 'Pearson r'},
            annot_kws={'size': 7}, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
savefig(fig, "fig05_sensor_correlation")
corr.round(4).to_csv(TAB_DIR / "eda_sensor_correlation.csv")

print(f"CO2-TVOC Pearson r = {r:.4f} (p={p:.1e}, n={len(pair):,})")
print("Figures written:", sorted(q.name for q in FIG_DIR.glob('fig0*')))
print("02_eda complete.")


CO2-TVOC Pearson r = 0.8422 (p=0.0e+00, n=109,819)
Figures written: ['fig01_label_distribution.pdf', 'fig01_label_distribution.png', 'fig02_lag_cross_correlation.pdf', 'fig02_lag_cross_correlation.png', 'fig03_risk_by_hour.pdf', 'fig03_risk_by_hour.png', 'fig04_co2_tvoc.pdf', 'fig04_co2_tvoc.png', 'fig05_sensor_correlation.pdf', 'fig05_sensor_correlation.png']
02_eda complete.
